In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from PIL import Image
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
OUTPUT_DIR=FIGURES
FIG_DPI=800
MODELS = [
    "cnn_lstm", "cnn_transformer", "unet", "transunet",
    "st_transunet", "da_transunet", "ta_transunet",
]
MODEL_LABELS = {
    "cnn_lstm": "CNN-LSTM",
    "cnn_transformer": "CNN-Transformer",
    "unet": "U-Net",
    "transunet": "TransUNet",
    "st_transunet": "ST-TransUNet",
    "da_transunet": "DA-TransUNet",
    "ta_transunet": "TA-TransUNet",
}
LEADS = [1, 2, 3, 4, 5, 6]
VARIABLES = ["u", "v", "w"]
VARIABLE_LABELS = {"u": "u", "v": "v", "w": "ω"}
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 22,
    "axes.titlesize": 26,
    "axes.labelsize": 24,
    "xtick.labelsize": 21,
    "ytick.labelsize": 21,
    "legend.fontsize": 18,
    "figure.titlesize": 28,
})

CACHE = RESULTS / 'backbone_metric_comparison_parallel32_gridwise'
BXLS = CACHE / 'Supplementary_backbone_R_RMSPE.xlsx'
SXLS = CACHE / 'Supplementary_backbone_u_v_omega_metrics.xlsx'
df=pd.read_excel(BXLS,sheet_name='Long_format')
if len(df)!=42 or df.duplicated(['Forecast lead time (h)','Model']).any():raise RuntimeError('Invalid saved main plot table')
main_summary=df.rename(columns={'Forecast lead time (h)':'lead_h','Model':'model_label','R':'R_mean','RMSPE':'RMSPE_mean'})
scaled=pd.read_excel(SXLS,sheet_name='Scaled_long')
if len(scaled)!=84 or scaled.duplicated(['Variable','Metric','Model']).any():raise RuntimeError('Invalid saved component plot table')
radar_scale_df=pd.read_excel(SXLS,sheet_name='Scale_factors')
inv_models={v:k for k,v in MODEL_LABELS.items()};inv_vars={v:k for k,v in VARIABLE_LABELS.items()}
scaled['model']=scaled['Model'].map(inv_models);scaled['variable']=scaled['Variable'].map(inv_vars)
if scaled[['model','variable']].isna().any().any():raise RuntimeError('Unknown saved plot label')
wide=scaled.pivot(index=['model','variable'],columns='Metric',values='Scaled value').reset_index()
radar_scaled=wide.rename(columns={'1-R':'scaled_one_minus_R','RMSE':'scaled_RMSE','MAE':'scaled_MAE','RMSPE':'scaled_RMSPE'})


In [ ]:
MAIN_BAR_PNG = OUTPUT_DIR / "Fig_backbone_R_RMSPE_bar_best.png"
MAIN_BAR_PDF = OUTPUT_DIR / "Fig_backbone_R_RMSPE_bar_best.pdf"


pivot_R = (
    main_summary
    .pivot(
        index="lead_h",
        columns="model_label",
        values="R_mean"
    )
    .reindex(index=LEADS)
)

pivot_RMSPE = (
    main_summary
    .pivot(
        index="lead_h",
        columns="model_label",
        values="RMSPE_mean"
    )
    .reindex(index=LEADS)
)

model_order_labels = [
    MODEL_LABELS[m]
    for m in MODELS
]

pivot_R = pivot_R[model_order_labels]
pivot_RMSPE = pivot_RMSPE[model_order_labels]


n_models = len(model_order_labels)

x = np.arange(len(LEADS))

bar_width = 0.11


R_YMIN = 0.70
RMSPE_YMIN = 0.040

R_DATA_MAX = float(
    np.nanmax(pivot_R.to_numpy(float))
)

RMSPE_DATA_MAX = float(
    np.nanmax(pivot_RMSPE.to_numpy(float))
)

R_YMAX = R_DATA_MAX + 0.020

RMSPE_YMAX = RMSPE_DATA_MAX + 0.0035


fig, axes = plt.subplots(
    1,
    2,
    figsize=(19.0, 8.8),
    constrained_layout=False
)


for i, model_label in enumerate(model_order_labels):

    values = pivot_R[model_label].to_numpy(float)

    axes[0].bar(
        x + (i - (n_models - 1) / 2) * bar_width,
        values - R_YMIN,
        bottom=R_YMIN,
        width=bar_width,
        label=model_label
    )


R_star_offset = 0.0045

for j, lead in enumerate(LEADS):

    vals = pivot_R.loc[lead].to_numpy(float)

    best_value = np.nanmax(vals)

    best_indices = np.where(
        np.isclose(
            vals,
            best_value,
            rtol=1e-10,
            atol=1e-12
        )
    )[0]

    for i in best_indices:

        xpos = (
            x[j] +
            (i - (n_models - 1) / 2)
            * bar_width
        )

        axes[0].scatter(
            xpos,
            best_value + R_star_offset,
            marker="*",
            s=180,
            color="black",
            zorder=10,
            clip_on=False
        )


axes[0].set_title(
    "(a) Pearson correlation coefficient",
    fontsize=27,
    fontweight="bold",
    pad=14
)

axes[0].set_xlabel(
    "Forecast lead time (h)",
    fontsize=24,
    labelpad=10
)

axes[0].set_ylabel(
    "R",
    fontsize=25,
    labelpad=10
)

axes[0].set_xticks(x)

axes[0].set_xticklabels(
    LEADS
)

axes[0].set_ylim(
    R_YMIN,
    R_YMAX
)

axes[0].tick_params(
    axis="both",
    labelsize=21,
    pad=6
)

axes[0].grid(
    True,
    axis="y",
    alpha=0.25
)

axes[0].set_axisbelow(True)


for i, model_label in enumerate(model_order_labels):

    values = (
        pivot_RMSPE[model_label]
        .to_numpy(float)
    )

    axes[1].bar(
        x + (i - (n_models - 1) / 2) * bar_width,
        values - RMSPE_YMIN,
        bottom=RMSPE_YMIN,
        width=bar_width,
        label=model_label
    )


RMSPE_star_offset = 0.00075

for j, lead in enumerate(LEADS):

    vals = (
        pivot_RMSPE
        .loc[lead]
        .to_numpy(float)
    )

    best_value = np.nanmin(vals)

    best_indices = np.where(
        np.isclose(
            vals,
            best_value,
            rtol=1e-10,
            atol=1e-12
        )
    )[0]

    for i in best_indices:

        xpos = (
            x[j] +
            (i - (n_models - 1) / 2)
            * bar_width
        )

        axes[1].scatter(
            xpos,
            best_value + RMSPE_star_offset,
            marker="*",
            s=180,
            color="black",
            zorder=10,
            clip_on=False
        )


axes[1].set_title(
    "(b) RMSPE",
    fontsize=27,
    fontweight="bold",
    pad=14
)

axes[1].set_xlabel(
    "Forecast lead time (h)",
    fontsize=24,
    labelpad=10
)

axes[1].set_ylabel(
    "RMSPE",
    fontsize=25,
    labelpad=10
)

axes[1].set_xticks(x)

axes[1].set_xticklabels(
    LEADS
)

axes[1].set_ylim(
    RMSPE_YMIN,
    RMSPE_YMAX
)

axes[1].tick_params(
    axis="both",
    labelsize=21,
    pad=6
)

axes[1].grid(
    True,
    axis="y",
    alpha=0.25
)

axes[1].set_axisbelow(True)


handles, labels = (
    axes[0].get_legend_handles_labels()
)

fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=4,
    frameon=False,
    fontsize=19,
    columnspacing=1.8,
    handlelength=1.8
)


fig.subplots_adjust(
    left=0.07,
    right=0.99,
    top=0.88,
    bottom=0.26,
    wspace=0.25
)


fig.savefig(
    MAIN_BAR_PNG,
    dpi=FIG_DPI,
    bbox_inches="tight",
    pad_inches=0.08
)

fig.savefig(
    MAIN_BAR_PDF,
    bbox_inches="tight",
    pad_inches=0.08
)

plt.show()

print(
    "正文柱状图已保存：",
    MAIN_BAR_PNG
)


In [ ]:
RADAR_BAR_PNG = (
    OUTPUT_DIR /
    "Fig_backbone_scaled_bar_u_v_omega_best_large_font.png"
)

RADAR_BAR_PDF = (
    OUTPUT_DIR /
    "Fig_backbone_scaled_bar_u_v_omega_best_large_font.pdf"
)


metric_key_order = [
    "scaled_one_minus_R",
    "scaled_RMSE",
    "scaled_MAE",
    "scaled_RMSPE"
]

metric_display_order = [
    "1-R",
    "RMSE",
    "MAE",
    "RMSPE"
]


metric_labels_by_variable = {}

for variable in VARIABLES:

    tmp = radar_scale_df.loc[
        radar_scale_df["Variable"] ==
        VARIABLE_LABELS[variable]
    ].copy()

    formula_map = {
        row["Metric"]: row["Formula"]
        for _, row in tmp.iterrows()
    }

    metric_labels_by_variable[variable] = [
        formula_map[m]
        for m in metric_display_order
    ]


def get_compact_bar_ylim(values):
    """
    为每个变量单独确定紧凑的y轴范围。

    下限：
    在实际最小值下方留约15%数据跨度，
    并向下取整到0.05。

    上限：
    在实际最大值上方留足星号空间，
    并向上取整到0.05。
    """

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    vmin = float(
        np.nanmin(values)
    )

    vmax = float(
        np.nanmax(values)
    )

    vrange = vmax - vmin

    if vrange == 0:
        vrange = max(
            abs(vmax) * 0.1,
            0.1
        )

    raw_ymin = (
        vmin -
        vrange * 0.15
    )

    raw_ymax = (
        vmax +
        vrange * 0.20
    )

    step = 0.05

    ymin = (
        np.floor(
            raw_ymin / step
        ) * step
    )

    ymax = (
        np.ceil(
            raw_ymax / step
        ) * step
    )

    ymin = max(
        0.0,
        ymin
    )

    return (
        float(ymin),
        float(ymax)
    )


X_LABEL_FONTSIZE = 30
PANEL_TITLE_FONTSIZE = 40
Y_LABEL_FONTSIZE = 36
Y_TICK_FONTSIZE = 30
SUPTITLE_FONTSIZE = 42
LEGEND_FONTSIZE = 30


fig, axes = plt.subplots(
    1,
    3,
    figsize=(30.0, 11.5),
    constrained_layout=False
)

letters = [
    "(a)",
    "(b)",
    "(c)"
]

n_models = len(MODELS)

x = np.arange(
    len(metric_key_order)
)

bar_width = 0.11


for ax, variable, letter in zip(
    axes,
    VARIABLES,
    letters
):


    sub = radar_scaled.loc[
        radar_scaled.variable ==
        variable
    ].copy()

    sub["model"] = pd.Categorical(
        sub["model"],
        categories=MODELS,
        ordered=True
    )

    sub = sub.sort_values(
        "model"
    )


    mat = []

    for model in MODELS:

        row = sub.loc[
            sub.model == model
        ]

        vals = [
            float(
                row.iloc[0][k]
            )
            for k in metric_key_order
        ]

        mat.append(vals)

    mat = np.array(
        mat,
        dtype=float
    )


    ymin, ymax = (
        get_compact_bar_ylim(mat)
    )

    visible_range = (
        ymax - ymin
    )


    for i, model in enumerate(MODELS):

        xpos = (
            x +
            (i - (n_models - 1) / 2)
            * bar_width
        )

        ax.bar(
            xpos,
            mat[i] - ymin,
            bottom=ymin,
            width=bar_width,
            label=MODEL_LABELS[model]
        )


    star_offset = (
        visible_range * 0.030
    )

    for j in range(
        len(metric_key_order)
    ):

        metric_values = mat[:, j]

        best_value = np.nanmin(
            metric_values
        )

        best_indices = np.where(
            np.isclose(
                metric_values,
                best_value,
                rtol=1e-10,
                atol=1e-12
            )
        )[0]

        for i in best_indices:

            xpos = (
                x[j] +
                (i - (n_models - 1) / 2)
                * bar_width
            )

            ax.scatter(
                xpos,
                best_value + star_offset,
                marker="*",
                s=320,
                color="black",
                zorder=10,
                clip_on=False
            )


    ax.set_xticks(x)

    ax.set_xticklabels(
        metric_labels_by_variable[
            variable
        ],
        fontsize=X_LABEL_FONTSIZE
    )

    ax.tick_params(
        axis="x",
        pad=8
    )


    ax.set_title(
        f"{letter} "
        f"{VARIABLE_LABELS[variable]}",
        fontsize=PANEL_TITLE_FONTSIZE,
        fontweight="bold",
        pad=18
    )


    ax.set_ylabel(
        "Scaled value",
        fontsize=Y_LABEL_FONTSIZE,
        labelpad=14
    )

    ax.set_ylim(
        ymin,
        ymax
    )

    ax.tick_params(
        axis="y",
        labelsize=Y_TICK_FONTSIZE,
        pad=8
    )


    ax.grid(
        True,
        axis="y",
        alpha=0.25,
        linewidth=1.2
    )

    ax.set_axisbelow(True)


fig.suptitle(
    "Backbone performance "
    "(scaled error-oriented metrics; higher = worse)",
    fontsize=SUPTITLE_FONTSIZE,
    fontweight="bold",
    y=0.985
)


handles, labels = (
    axes[0].get_legend_handles_labels()
)

fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.018),
    ncol=4,
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
    columnspacing=2.0,
    handlelength=2.0,
    handletextpad=0.8
)


fig.subplots_adjust(
    left=0.060,
    right=0.995,
    top=0.82,
    bottom=0.29,
    wspace=0.28
)


fig.savefig(
    RADAR_BAR_PNG,
    dpi=FIG_DPI,
    bbox_inches="tight",
    pad_inches=0.12
)

fig.savefig(
    RADAR_BAR_PDF,
    bbox_inches="tight",
    pad_inches=0.12
)

plt.show()

print(
    "大字体柱状图已保存：",
    RADAR_BAR_PNG
)

print(
    "PDF版本已保存：",
    RADAR_BAR_PDF
)
